In [43]:
import pandas as pd 

data = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
test_ids = test["PassengerId"]

In [44]:


print(data.nunique())

PassengerId    891
Survived         2
Pclass           3
Name           891
Sex              2
Age             88
SibSp            7
Parch            7
Ticket         681
Fare           248
Cabin          147
Embarked         3
dtype: int64


In [45]:
first_class = data[data["Pclass"]==1]
second_class = data[data["Pclass"]==2]
third_class = data[data["Pclass"]==3]

In [46]:
print(first_class["Fare"].mean())


print(second_class["Fare"].mean())
print(third_class["Fare"].mean())

84.1546875
20.662183152173913
13.675550101832993


In [47]:
print("fraction who survived in first class")
print(len(first_class[first_class["Survived"]==1])/len(first_class))

print("fraction who survived in second class")
print(len(second_class[second_class["Survived"]==1])/len(second_class))

print("fraction who survived in third class")
print(len(third_class[third_class["Survived"]==1])/len(third_class))


fraction who survived in first class
0.6296296296296297
fraction who survived in second class
0.47282608695652173
fraction who survived in third class
0.24236252545824846


In [48]:
def clean(data):
    data = data.drop(["Ticket","Cabin", "Name", "PassengerId"], axis = 1)

    cols = ["SibSp", "Parch", "Fare", "Age"]
    for col in cols:
        data[col] = data[col].fillna(data[col].median())

        data["Embarked"] = data["Embarked"].fillna("U")
        #fill unknown values with na
        # we can try and infer mean of ages by seeing if first class, (generally older for age), and so on stuff like that 
    return data

data = clean(data)
test = clean(test)

In [49]:
data.head(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [ ]:
from sklearn import preprocessing 

le = preprocessing.LabelEncoder()

cols = ["Sex"]

for col in cols:
    data[col] = le.fit_transform(data[col])
    test[col] = le.transform(test[col])

data = pd.get_dummies(data, columns=["Embarked"], dtype=int)
test = pd.get_dummies(test, columns=["Embarked"], dtype=int)

test = test.reindex(columns=data.drop("Survived", axis=1).columns, fill_value=0)

data.head(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S,Embarked_U
0,0,3,1,22.0,1,0,7.2500,0,0,1,0
1,1,1,0,38.0,1,0,71.2833,1,0,0,0
2,1,3,0,26.0,0,0,7.9250,0,0,1,0
3,1,1,0,35.0,1,0,53.1000,0,0,1,0
4,0,3,1,35.0,0,0,8.0500,0,0,1,0


In [51]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

y = data["Survived"]
x = data.drop("Survived",axis=1)

x_train,x_val,y_train,y_val = train_test_split(x,y,test_size = 0.2, random_state = 42)

In [52]:
clf = LogisticRegression(random_state=0, max_iter=1000 ).fit(x_train,y_train)

In [53]:
print(clf.feature_names_in_)
print(test.columns)

['Pclass' 'Sex' 'Age' 'SibSp' 'Parch' 'Fare' 'Embarked_C' 'Embarked_Q'
 'Embarked_S' 'Embarked_U']
Index(['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_C',
       'Embarked_Q', 'Embarked_S'],
      dtype='str')


In [54]:
predictions = clf.predict(x_val)

from sklearn.metrics import accuracy_score
accuracy_score(y_val,predictions)

0.8100558659217877

In [55]:
submission_preds = clf.predict(test)

df = pd.DataFrame({"PassengerId":test_ids.values, "Survived" : submission_preds })

ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- Embarked_U


In [ ]:
df.to_csv("submission.csv", index = False)